## inventory

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "slcf_forcings").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not locate src/slcf_forcings above cwd")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/563/as8561/side_projects/slcf_forcings


In [2]:
from slcf_forcings.paths import CEDS, QV56
from slcf_forcings.ceds import discover_emissions_files

print(QV56.is_dir())
for label, path in CEDS.items():
    print(f"{label:12s} {path.is_dir()}  {path}")

files = discover_emissions_files(CEDS["production"], grid_label="gn", version="v20250421")
print(f"\nproduction gn v20250421: {len(files)} files")

True
production   True  /g/data/qv56/replicas/input4MIPs/CMIP7/CMIP/PNNL-JGCRI/CEDS-CMIP-2025-04-18
archive      True  /g/data/qv56/replicas/input4MIPs/CMIP7/CMIP/PNNL-JGCRI/CEDS-CMIP-2025-03-18
preprod_nov  True  /g/data/qv56/replicas/input4MIPs/CMIP6Plus/CMIP/PNNL-JGCRI/CEDS-CMIP-2024-11-25
preprod_oct  True  /g/data/qv56/replicas/input4MIPs/CMIP6Plus/CMIP/PNNL-JGCRI/CEDS-CMIP-2024-10-21
cmip6        True  /g/data/qv56/replicas/input4MIPs/CMIP6/CMIP/PNNL-JGCRI/CEDS-2017-05-18

production gn v20250421: 104 files


In [ ]:
from slcf_forcings.ceds import audit_collection, check_single_realisation
from slcf_forcings.paths import CEDS

march = audit_collection(CEDS["archive"], grid_label="gn")

print(check_single_realisation(march).to_string(index=False))
print()
print(
    march[march["version_path"] == "v20250327"]
    [["species", "stream", "chunk_first_year", "chunk_last_year", "size_bytes"]]
    .sort_values(["species", "chunk_first_year"])
    .to_string(index=False)
)

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.10/lib/python3.12/site-packages/argopy/utils/lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'requests.exceptions.ConnectionError'>
HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /IrishMarineInstitute/awesome-erddap/master/erddaps.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fad68a74b90>: Failed to establish a new connection: [Errno 101] Network is unreachable'))
  warnings.warn(
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.10/lib/python3.12/site-packages/argopy/utils/lists.py:50: UserWarning: An error occurred while loading the ArgoVis data fetcher, it will not be available !
<class 'requests.exceptions.ConnectionError'>
HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /IrishMarineInstitute/awesome-erddap/master/erddaps.jso

  source_id_filename grid_label_path version_path  n_files    gb
CEDS-CMIP-2025-03-18              gn    v20250325       96 47.38
CEDS-CMIP-2025-03-18              gn    v20250327        8  4.06

species  stream  chunk_first_year  chunk_last_year  size_bytes
     CO surface              1750             1799   189649948
     CO surface              1800             1849   375075050
     CO surface              1850             1899   590753969
     CO surface              1900             1949   691124271
     CO surface              1950             1999   726390115
     CO surface              2000             2023   364398805
    N2O surface              1970             2019  1039691795
    N2O surface              2020             2023    84378895


In [ ]:
import xarray as xr
from slcf_forcings.ceds import audit_collection
from slcf_forcings.paths import CEDS

n2o_new = march[(march["species"] == "N2O") & (march["version_path"] == "v20250327")]
with xr.open_dataset(n2o_new.iloc[0]["path"], decode_times=False) as ds:
    for key in ("source_version", "comment", "history", "further_info_url",
                "data_specs_version", "creation_date", "tracking_id"):
        if key in ds.attrs:
            print(f"{key}: {ds.attrs[key]}")

source_version: 2025-03-18
comment: Research data produced using the Community Emissions Data System (CEDS)
history: 25-03-2025 21:45:03 PM UTC; College Park, MD, 20742
further_info_url: https://www.pnnl.gov/projects/ceds
creation_date: 2025-03-25T21:45:03Z
tracking_id: hdl:21.14100/68f57686-09c2-11f0-9c39-3cecef4e749a


In [ ]:
from pathlib import Path
for p in CEDS["archive"].glob("**/areacella*.nc"):
    print(p.relative_to(CEDS["archive"]))

atmos/fx/areacella/gn/v20250325/areacella_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn.nc


In [ ]:
for species in ("CO", "N2O"):
    print(f"\n{species}:")
    for p in sorted((CEDS["archive"]).glob(f"**/{species}_em_anthro/gn/*/*.nc")):
        print("  ", p.relative_to(CEDS["archive"]))


CO:
   atmos/mon/CO_em_anthro/gn/v20250327/CO-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_175001-179912.nc
   atmos/mon/CO_em_anthro/gn/v20250327/CO-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_180001-184912.nc
   atmos/mon/CO_em_anthro/gn/v20250327/CO-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_185001-189912.nc
   atmos/mon/CO_em_anthro/gn/v20250327/CO-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_190001-194912.nc
   atmos/mon/CO_em_anthro/gn/v20250327/CO-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_195001-199912.nc
   atmos/mon/CO_em_anthro/gn/v20250327/CO-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_200001-202312.nc

N2O:
   atmos/mon/N2O_em_anthro/gn/v20250327/N2O-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_197001-201912.nc
   atmos/mon/N2O_em_anthro/gn/v20250327/N2O-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-03-18_gn_202001-202312.nc


In [ ]:
from slcf_forcings.ceds import sha256_of
# pick one matching pair from the listing above
a = CEDS["archive"] / "atmos/mon/CO_em_anthro/gn/v20250325/<file>.nc"
b = CEDS["archive"] / "atmos/mon/CO_em_anthro/gn/v20250327/<file>.nc"
print(sha256_of(a) == sha256_of(b))

FileNotFoundError: [Errno 2] No such file or directory: '/g/data/qv56/replicas/input4MIPs/CMIP7/CMIP/PNNL-JGCRI/CEDS-CMIP-2025-03-18/atmos/mon/CO_em_anthro/gn/v20250325/<file>.nc'

In [ ]:
from slcf_forcings.ceds import select_files
from slcf_forcings.paths import CEDS

march = select_files(CEDS["archive"], allow_multiple_versions=True)
print(f"archive: {len(march)} files, {len(set(p.name.split('-em-')[0] for p in march))} species")

april = select_files(CEDS["production"], version="v20250421")
print(f"production: {len(april)} files, {len(set(p.name.split('-em-')[0] for p in april))} species")

archive: 52 files, 10 species
production: 52 files, 10 species


In [ ]:
from slcf_forcings.ceds import (
    audit_collection, assert_all_readable, check_source_consistency,
    check_time_coverage, check_single_realisation, write_manifest,
)
from slcf_forcings.paths import CEDS, VERSIONS

OUT = PROJECT_ROOT / "output" / "production"
OUT.mkdir(parents=True, exist_ok=True)

audit = audit_collection(CEDS["production"], grid_label="gn", version="v20250421")
audit.to_csv(OUT / "audit.csv", index=False)

print(f"{len(audit)} files\n")
print(check_single_realisation(audit).to_string(index=False))
print(f"\nUnreadable:          {(~audit['readable'].fillna(False)).sum()}")
print(f"source_id mismatch:  {len(check_source_consistency(audit))}")
print(f"time coverage bad:   {len(check_time_coverage(audit))}")
print(f"Units:               {sorted(audit['units'].dropna().unique())}")
print(f"Sector counts:       {sorted(audit.loc[audit['stream']=='surface','n_sectors'].unique())}")

assert_all_readable(audit, context="production gn v20250421")
print("\nPASS")

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.10/lib/python3.12/site-packages/argopy/utils/lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'requests.exceptions.ConnectionError'>
HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /IrishMarineInstitute/awesome-erddap/master/erddaps.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f81bf1e2120>: Failed to establish a new connection: [Errno 101] Network is unreachable'))
  warnings.warn(
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.10/lib/python3.12/site-packages/argopy/utils/lists.py:50: UserWarning: An error occurred while loading the ArgoVis data fetcher, it will not be available !
<class 'requests.exceptions.ConnectionError'>
HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /IrishMarineInstitute/awesome-erddap/master/erddaps.jso

104 files

  source_id_filename grid_label_path version_path  n_files    gb
CEDS-CMIP-2025-04-18              gn    v20250421      104 51.47

Unreadable:          0
source_id mismatch:  0
time coverage bad:   0
Units:               ['kg m-2 s-1']
Sector counts:       [np.int64(8)]

PASS


In [ ]:
manifest = write_manifest(
    CEDS["production"], OUT / "data_manifest.csv",
    grid_label="gn", version="v20250421", compute_checksums=True,
)
print(f"{len(manifest)} rows")
print(manifest[["file", "source_id", "version", "first_year", "last_year",
                "n_months", "license"]].head().to_string(index=False))

Manifest:   0%|          | 0/104 [00:00<?, ?file/s]

104 rows
                                                                               file            source_id   version  first_year  last_year  n_months                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   license
BC-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-04-18_gn_175001-179912.nc CEDS-CMIP-2025-04-18 v20250421        1750       1799       600 The input4MIPs data linked to this entry is l

In [ ]:
ok = audit[audit["readable"].fillna(False).astype(bool)].copy()
print(ok[["chunk_first_year", "chunk_last_year", "time_size"]].dtypes)
exp = (ok["chunk_last_year"] - ok["chunk_first_year"] + 1) * 12
print(exp.head())
print(ok["time_size"].head())
print((ok["time_size"] != exp).head())

chunk_first_year    int64
chunk_last_year     int64
time_size           int64
dtype: object
0    600
1    600
2    600
3    600
4    600
dtype: int64
0    600
1    600
2    600
3    600
4    600
Name: time_size, dtype: int64
0    False
1    False
2    False
3    False
4    False
dtype: bool


In [ ]:
from slcf_forcings.ceds import sector_labels, select_files
from slcf_forcings.paths import CEDS

f = select_files(CEDS["production"], ["SO2"], (2000, 2010), version="v20250421")[0]
labels = sector_labels(f)
print(labels if labels else "EMPTY — parser failed")

if not labels:
    import xarray as xr
    with xr.open_dataset(f, decode_times=False) as ds:
        print(dict(ds["sector"].attrs))

{0: 'Agriculture', 1: 'Energy', 2: 'Industrial', 3: 'Transportation', 4: 'Residential, Commercial, Other', 5: 'Solvents production and application', 6: 'Waste', 7: 'International Shipping'}


In [ ]:
from slcf_forcings.ceds import compare_sector_definitions, select_files
from slcf_forcings.paths import CEDS

a = select_files(CEDS["cmip6"], ["SO2"], (2000, 2010))[0]
b = select_files(CEDS["production"], ["SO2"], (2000, 2010), version="v20250421")[0]

print(f"CMIP6: {a.name}")
print(f"CMIP7: {b.name}\n")
print(compare_sector_definitions(a, b).to_string(index=False))

CMIP6: SO2-em-anthro_input4MIPs_emissions_CMIP_CEDS-2017-05-18_gn_200001-201412.nc
CMIP7: SO2-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-04-18_gn_200001-202312.nc

 index                                left                               right  same
     0                         Agriculture                         Agriculture  True
     1                              Energy                              Energy  True
     2                          Industrial                          Industrial  True
     3                      Transportation                      Transportation  True
     4      Residential, Commercial, Other      Residential, Commercial, Other  True
     5 Solvents production and application Solvents production and application  True
     6                               Waste                               Waste  True
     7              International Shipping              International Shipping  True


In [ ]:
from itertools import combinations

refs = {}
for label in ["cmip6", "preprod_oct", "preprod_nov", "archive", "production"]:
    try:
        f = select_files(CEDS[label], ["SO2"], (2000, 2010),
                         allow_multiple_versions=True)[0]
        refs[label] = sector_labels(f)
    except Exception as e:
        print(f"{label}: {type(e).__name__} — {e}")

for a, b in combinations(refs, 2):
    same = refs[a] == refs[b]
    print(f"{a:12s} vs {b:12s} {'identical' if same else 'DIFFERENT'}")

preprod_oct: ValueError — No files covering 2000-2010 for: ['SO2'] in /g/data/qv56/replicas/input4MIPs/CMIP6Plus/CMIP/PNNL-JGCRI/CEDS-CMIP-2024-10-21
cmip6        vs preprod_nov  identical
cmip6        vs archive      identical
cmip6        vs production   identical
preprod_nov  vs archive      identical
preprod_nov  vs production   identical
archive      vs production   identical


In [3]:
from slcf_forcings.ceds import audit_collection
a = audit_collection(CEDS["preprod_oct"])
print(a.groupby(["species", "stream"]).agg(n=("file","count"),
      y0=("chunk_first_year","min"), y1=("chunk_last_year","max")).to_string())

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.10/lib/python3.12/site-packages/argopy/utils/lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'requests.exceptions.ConnectionError'>
HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /IrishMarineInstitute/awesome-erddap/master/erddaps.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fd9e5765bb0>: Failed to establish a new connection: [Errno 101] Network is unreachable'))
  warnings.warn(
/g/data/xp65/public/apps/med_conda/envs/analysis3-26.10/lib/python3.12/site-packages/argopy/utils/lists.py:50: UserWarning: An error occurred while loading the ArgoVis data fetcher, it will not be available !
<class 'requests.exceptions.ConnectionError'>
HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /IrishMarineInstitute/awesome-erddap/master/erddaps.jso

                 n    y0    y1
species stream                
BC      AIR      6  1750  2022
CH4     AIR      2  1970  2022
CO      AIR      6  1750  2022
CO2     AIR      6  1750  2022
N2O     AIR      2  1970  2022
NH3     AIR      6  1750  2022
NMVOC   AIR      6  1750  2022
        surface  6  1750  2022
NOx     AIR      6  1750  2022
OC      AIR      6  1750  2022
SO2     AIR      6  1750  2022


In [4]:
a = audit_collection(CEDS["preprod_nov"])
print(a.groupby(["species", "stream"]).agg(n=("file","count"),
      y0=("chunk_first_year","min"), y1=("chunk_last_year","max")).to_string())

                 n    y0    y1
species stream                
BC      surface  6  1750  2022
CH4     surface  2  1970  2022
CO      surface  6  1750  2022
CO2     surface  6  1750  2022
N2O     surface  2  1970  2022
NH3     surface  6  1750  2022
NOx     surface  6  1750  2022
OC      surface  6  1750  2022
SO2     surface  6  1750  2022
